In [1]:
"""
inspect_datasets.py
-------------------
Inspect the folder structure, file types, and contents of the 4 bearing datasets:
  - CWRU
  - HUST bearing dataset
  - MFPT Fault Data Sets
  - Paderborn

Run from any Python env with: numpy, scipy
(For HUST it may also have .csv or .mat — script handles both.)

Usage:
    python inspect_datasets.py

Output:
    Prints a structured report to stdout AND saves it to
    F:\\Umar-Wisal-Work\\Datasets\\dataset_inspection_report.txt
"""

import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

# Optional imports — fall back gracefully
try:
    import scipy.io as sio
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("[warn] scipy not installed — .mat inspection will be skipped")

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

# ──────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────
ROOT = Path(r"F:\Umar-Wisal-Work\Datasets")

DATASETS = {
    "CWRU":     ROOT / "CWRU",
    "HUST":     ROOT / "HUST bearing dataset",
    "MFPT":     ROOT / "MFPT Fault Data Sets",
    "Paderborn": ROOT / "Paderborn",
}

REPORT_PATH = ROOT / "dataset_inspection_report.txt"

# How many sample filenames to print per folder
N_SAMPLE_FILES = 8
# How many sample folders to print per dataset (for deep hierarchies like Paderborn)
N_SAMPLE_SUBDIRS = 6
# Max depth to walk
MAX_DEPTH = 6
# How many .mat files to peek inside per dataset
N_MAT_PEEKS = 3


# ──────────────────────────────────────────────────────────────────────────
# OUTPUT BUFFER — capture everything so we save AND print
# ──────────────────────────────────────────────────────────────────────────
class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, s):
        for st in self.streams:
            st.write(s)
            st.flush()
    def flush(self):
        for st in self.streams:
            st.flush()


# ──────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────
def header(title, char="=", width=78):
    print()
    print(char * width)
    print(title)
    print(char * width)


def subheader(title, char="-", width=78):
    print()
    print(char * width)
    print(title)
    print(char * width)


def walk_tree(root, max_depth=MAX_DEPTH):
    """
    Walk a directory tree up to max_depth and yield (depth, path, is_dir).
    """
    root = Path(root)
    if not root.exists():
        return
    root_depth = len(root.parts)
    for path in root.rglob("*"):
        depth = len(path.parts) - root_depth
        if depth > max_depth:
            continue
        yield depth, path, path.is_dir()


def summarize_extensions(root):
    """Count file extensions under root."""
    counter = Counter()
    sizes = defaultdict(list)
    for _, p, is_dir in walk_tree(root):
        if not is_dir:
            ext = p.suffix.lower() if p.suffix else "(no ext)"
            counter[ext] += 1
            try:
                sizes[ext].append(p.stat().st_size)
            except OSError:
                pass
    return counter, sizes


def list_top_level_dirs(root, max_show=20):
    """List immediate subdirectories of root."""
    if not root.exists():
        return []
    return sorted([p for p in root.iterdir() if p.is_dir()])[:max_show]


def folder_tree_summary(root, max_depth=3):
    """Print a compact tree showing folder structure (dirs only) up to max_depth."""
    if not root.exists():
        print(f"  [missing] {root}")
        return
    root_depth = len(root.parts)
    for path in sorted(root.rglob("*")):
        if not path.is_dir():
            continue
        depth = len(path.parts) - root_depth
        if depth > max_depth or depth == 0:
            continue
        indent = "  " * depth
        # Count files directly inside this folder
        try:
            n_files = sum(1 for x in path.iterdir() if x.is_file())
            n_subdirs = sum(1 for x in path.iterdir() if x.is_dir())
        except (PermissionError, OSError):
            n_files = n_subdirs = -1
        print(f"  {indent}└── {path.name}/  [files: {n_files}, subdirs: {n_subdirs}]")


def sample_filenames(folder, n=N_SAMPLE_FILES):
    """Return up to n filenames from a folder."""
    if not folder.exists():
        return []
    files = sorted([p.name for p in folder.iterdir() if p.is_file()])
    return files[:n]


def peek_mat_file(mat_path):
    """Open a .mat file and return variable names + shapes."""
    if not HAS_SCIPY:
        return None
    try:
        # First try standard scipy.io (works for v5 .mat files)
        try:
            data = sio.loadmat(str(mat_path), squeeze_me=False, struct_as_record=False)
        except NotImplementedError:
            # v7.3 .mat files need h5py
            try:
                import h5py
                with h5py.File(mat_path, "r") as f:
                    keys = list(f.keys())
                    info = {}
                    for k in keys:
                        try:
                            info[k] = {"shape": f[k].shape, "dtype": str(f[k].dtype)}
                        except Exception as e:
                            info[k] = {"error": str(e)}
                return {"format": "v7.3 (HDF5)", "vars": info}
            except ImportError:
                return {"error": "v7.3 .mat — install h5py to inspect"}

        info = {}
        for k, v in data.items():
            if k.startswith("__"):
                continue
            try:
                shape = getattr(v, "shape", None)
                dtype = getattr(v, "dtype", type(v).__name__)
                info[k] = {"shape": shape, "dtype": str(dtype)}
            except Exception as e:
                info[k] = {"error": str(e)}
        return {"format": "v5", "vars": info}
    except Exception as e:
        return {"error": str(e)}


def peek_csv_file(csv_path, n_lines=5):
    """Peek first few lines of a CSV."""
    try:
        with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
            lines = []
            for i, line in enumerate(f):
                if i >= n_lines:
                    break
                lines.append(line.rstrip())
        return lines
    except Exception as e:
        return [f"[error reading: {e}]"]


def find_files_by_ext(root, ext, limit=None):
    """Find files by extension under root."""
    if not root.exists():
        return []
    out = []
    for p in root.rglob(f"*{ext}"):
        if p.is_file():
            out.append(p)
            if limit and len(out) >= limit:
                break
    return out


# ──────────────────────────────────────────────────────────────────────────
# Dataset-specific inspectors
# ──────────────────────────────────────────────────────────────────────────
def inspect_dataset(name, path):
    header(f"DATASET: {name}")
    print(f"Path: {path}")

    if not path.exists():
        print(f"  [!!] Path does not exist. Skipping.")
        return

    # 1. Top-level structure
    subheader("Top-level structure (depth ≤ 3)")
    folder_tree_summary(path, max_depth=3)

    # 2. Extension summary
    subheader("File extension counts")
    counter, sizes = summarize_extensions(path)
    if not counter:
        print("  (no files found)")
    else:
        for ext, cnt in counter.most_common():
            avg_mb = (sum(sizes[ext]) / len(sizes[ext]) / 1024 / 1024) if sizes[ext] else 0
            print(f"  {ext:12s} : {cnt:6d} files   (avg size: {avg_mb:.2f} MB)")

    # 3. Sample filenames from a few folders
    subheader("Sample filenames (per subfolder)")
    subdirs = []
    for path_obj in path.rglob("*"):
        if path_obj.is_dir():
            # Only include folders that directly contain files
            try:
                has_files = any(p.is_file() for p in path_obj.iterdir())
            except (PermissionError, OSError):
                has_files = False
            if has_files:
                subdirs.append(path_obj)
        if len(subdirs) >= N_SAMPLE_SUBDIRS * 3:
            break

    # If no subdirs with files, sample from root itself
    if not subdirs:
        subdirs = [path]

    for sub in subdirs[:N_SAMPLE_SUBDIRS]:
        rel = sub.relative_to(path) if sub != path else Path(".")
        print(f"  📁 {rel}")
        for fn in sample_filenames(sub, n=N_SAMPLE_FILES):
            print(f"      {fn}")

    # 4. Peek inside sample .mat files
    mat_files = find_files_by_ext(path, ".mat", limit=N_MAT_PEEKS)
    if mat_files:
        subheader(f"Peek inside .mat files (first {len(mat_files)})")
        for mp in mat_files:
            rel = mp.relative_to(path)
            print(f"  📄 {rel}")
            info = peek_mat_file(mp)
            if info is None:
                print("      (scipy not available)")
            elif "error" in info:
                print(f"      [error] {info['error']}")
            else:
                print(f"      format: {info.get('format', '?')}")
                for vname, vinfo in info.get("vars", {}).items():
                    print(f"        - {vname}: {vinfo}")

    # 5. Peek inside sample .csv files
    csv_files = find_files_by_ext(path, ".csv", limit=2)
    if csv_files:
        subheader(f"Peek inside .csv files (first {len(csv_files)})")
        for cp in csv_files:
            rel = cp.relative_to(path)
            print(f"  📄 {rel}")
            for line in peek_csv_file(cp, n_lines=5):
                print(f"      {line}")

    # 6. Peek inside sample .txt files (some MFPT files use .txt)
    txt_files = find_files_by_ext(path, ".txt", limit=2)
    if txt_files:
        subheader(f"Peek inside .txt files (first {len(txt_files)})")
        for tp in txt_files:
            rel = tp.relative_to(path)
            print(f"  📄 {rel}")
            for line in peek_csv_file(tp, n_lines=5):
                print(f"      {line}")


# ──────────────────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────────────────
def main():
    # Open report file and tee output
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    report_file = open(REPORT_PATH, "w", encoding="utf-8")
    original_stdout = sys.stdout
    sys.stdout = Tee(original_stdout, report_file)

    try:
        header("BEARING DATASET STRUCTURE INSPECTION", char="#")
        print(f"Root folder: {ROOT}")
        print(f"Python: {sys.version.split()[0]}")
        print(f"scipy available: {HAS_SCIPY}")
        print(f"numpy available: {HAS_NUMPY}")

        for name, path in DATASETS.items():
            inspect_dataset(name, path)

        header("INSPECTION COMPLETE", char="#")
        print(f"Report saved to: {REPORT_PATH}")
    finally:
        sys.stdout = original_stdout
        report_file.close()

    print(f"\n✅ Done. Report saved to:\n   {REPORT_PATH}")
    print("Send me the contents of that file and I'll write the unified loader.")


if __name__ == "__main__":
    main()


##############################################################################
BEARING DATASET STRUCTURE INSPECTION
##############################################################################
Root folder: F:\Umar-Wisal-Work\Datasets
Python: 3.13.6
scipy available: True
numpy available: True

DATASET: CWRU
Path: F:\Umar-Wisal-Work\Datasets\CWRU

------------------------------------------------------------------------------
Top-level structure (depth ≤ 3)
------------------------------------------------------------------------------

------------------------------------------------------------------------------
File extension counts
------------------------------------------------------------------------------
  .mat         :      4 files   (avg size: 28.40 MB)

------------------------------------------------------------------------------
Sample filenames (per subfolder)
------------------------------------------------------------------------------
  📁 .
      ball.mat
      health

In [2]:
# inspect_bearing_datasets.py
# Run: python inspect_bearing_datasets.py

import os
import json
import csv
from pathlib import Path
import numpy as np
import scipy.io as sio

ROOTS = {
    "CWRU": r"F:\Umar-Wisal-Work\Datasets\CWRU",
    "HUST": r"F:\Umar-Wisal-Work\Datasets\HUST bearing dataset",
    "MFPT": r"F:\Umar-Wisal-Work\Datasets\MFPT Fault Data Sets",
    "Paderborn": r"F:\Umar-Wisal-Work\Datasets\Paderborn",
}

OUT_DIR = Path(r"F:\Umar-Wisal-Work\Datasets\_inspection")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def clean_keys(mat):
    return {k: v for k, v in mat.items() if not k.startswith("__")}


def safe_shape(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None


def is_mat_struct(x):
    return hasattr(x, "_fieldnames")


def summarize_value(x, depth=0, max_depth=4):
    if depth > max_depth:
        return {"type": "max_depth_reached"}

    if isinstance(x, np.ndarray):
        info = {
            "type": "ndarray",
            "shape": safe_shape(x),
            "dtype": str(x.dtype),
        }

        if x.dtype == object:
            info["object_preview"] = []
            flat = x.flatten()
            for i, item in enumerate(flat[:5]):
                info["object_preview"].append({
                    "index": i,
                    "summary": summarize_value(item, depth + 1, max_depth)
                })
        else:
            try:
                arr = np.asarray(x).astype(float)
                info.update({
                    "min": float(np.nanmin(arr)),
                    "max": float(np.nanmax(arr)),
                    "mean": float(np.nanmean(arr)),
                })
            except Exception:
                pass
        return info

    if is_mat_struct(x):
        return {
            "type": "mat_struct",
            "fields": {
                f: summarize_value(getattr(x, f), depth + 1, max_depth)
                for f in x._fieldnames
            }
        }

    if isinstance(x, (str, int, float, np.integer, np.floating)):
        return {"type": type(x).__name__, "value": str(x)[:200]}

    return {"type": type(x).__name__, "repr": repr(x)[:200]}


def infer_label(dataset, path):
    name = path.name.lower()
    parent = path.parent.name.lower()

    if dataset == "CWRU":
        if "healthy" in name:
            return "normal"
        if "inner" in name:
            return "inner"
        if "outer" in name:
            return "outer"
        if "ball" in name:
            return "ball"

    if dataset == "HUST":
        stem = path.stem.upper()
        if stem.startswith("N"):
            return "normal"
        if stem.startswith("I") and "O" not in stem and "B" not in stem:
            return "inner"
        if stem.startswith("O") and "B" not in stem:
            return "outer"
        if stem.startswith("B"):
            return "ball"
        if "IB" in stem:
            return "inner+ball"
        if "IO" in stem:
            return "inner+outer"
        if "OB" in stem:
            return "outer+ball"

    if dataset == "MFPT":
        p = str(path).lower()
        if "baseline" in p:
            return "normal"
        if "outer" in p:
            return "outer"
        if "inner" in p:
            return "inner"

    if dataset == "Paderborn":
        folder = path.parent.name.upper()
        if folder.startswith("K0") or folder.startswith("K00"):
            return "normal"
        if folder.startswith("KA"):
            return "outer"
        if folder.startswith("KI"):
            return "inner"

    return "unknown"


def collect_numeric_arrays(obj, prefix="", found=None):
    if found is None:
        found = []

    if isinstance(obj, np.ndarray):
        if obj.dtype != object and np.issubdtype(obj.dtype, np.number):
            found.append((prefix, obj.shape, str(obj.dtype)))
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                collect_numeric_arrays(item, f"{prefix}{idx}", found)

    elif is_mat_struct(obj):
        for f in obj._fieldnames:
            collect_numeric_arrays(getattr(obj, f), f"{prefix}.{f}" if prefix else f, found)

    return found


rows = []
full_report = {}

for dataset, root in ROOTS.items():
    root = Path(root)
    files = sorted(root.rglob("*.mat"))
    full_report[dataset] = {
        "root": str(root),
        "num_mat_files": len(files),
        "files": {}
    }

    print(f"\n{dataset}: {len(files)} .mat files")

    for path in files:
        rel = str(path.relative_to(root))
        label = infer_label(dataset, path)

        try:
            mat = clean_keys(sio.loadmat(path, squeeze_me=True, struct_as_record=False))
            keys = list(mat.keys())

            numeric_arrays = []
            for k, v in mat.items():
                numeric_arrays.extend(collect_numeric_arrays(v, k))

            largest = sorted(
                numeric_arrays,
                key=lambda x: np.prod(x[1]) if len(x[1]) > 0 else 0,
                reverse=True
            )[:5]

            full_report[dataset]["files"][rel] = {
                "label": label,
                "keys": keys,
                "largest_numeric_arrays": [
                    {"path": a[0], "shape": a[1], "dtype": a[2]} for a in largest
                ],
                "top_level_summary": {k: summarize_value(v, max_depth=3) for k, v in mat.items()}
            }

            rows.append({
                "dataset": dataset,
                "file": rel,
                "label": label,
                "keys": ";".join(keys),
                "largest_array_path": largest[0][0] if largest else "",
                "largest_array_shape": str(largest[0][1]) if largest else "",
                "largest_array_dtype": largest[0][2] if largest else "",
            })

            print(f"  OK  {rel} | label={label} | keys={keys}")

        except Exception as e:
            rows.append({
                "dataset": dataset,
                "file": rel,
                "label": label,
                "keys": "ERROR",
                "largest_array_path": "",
                "largest_array_shape": "",
                "largest_array_dtype": "",
            })
            full_report[dataset]["files"][rel] = {"error": str(e)}
            print(f"  ERR {rel}: {e}")

csv_path = OUT_DIR / "dataset_structure_summary.csv"
json_path = OUT_DIR / "dataset_structure_full_report.json"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(full_report, f, indent=2, default=str)

print("\nSaved:")
print(csv_path)
print(json_path)


CWRU: 4 .mat files
  OK  ball.mat | label=ball | keys=['Ball_CW']
  OK  healthy.mat | label=normal | keys=['Healthy_CW']
  OK  inner.mat | label=inner | keys=['Inner_CW']
  OK  outer.mat | label=outer | keys=['Outer_CW']

HUST: 99 .mat files
  OK  B500.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B502.mat | label=ball | keys=['data', 'fs', 'ru_raw']
  OK  B504.mat | label=ball | keys=['data', 'fs', 'ru_raw']
  OK  B600.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B602.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B604.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B700.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B702.mat | label=ball | keys=['data', 'fs', 'ru_raw']
  OK  B704.mat | label=ball | keys=['data', 'fs', 'ru_raw']
  OK  B800.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B802.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  B804.mat | label=ball | keys=['data', 'fs', 'rpm', 'ru']
  OK  I400.ma

In [3]:
# build_unified_benchmark.py

import os
import numpy as np
import pandas as pd
import scipy.io as sio
from pathlib import Path
from tqdm import tqdm

# ============================================================
# ROOT PATHS
# ============================================================

ROOTS = {
    "CWRU": r"F:\Umar-Wisal-Work\Datasets\CWRU",
    "HUST": r"F:\Umar-Wisal-Work\Datasets\HUST bearing dataset",
    "MFPT": r"F:\Umar-Wisal-Work\Datasets\MFPT Fault Data Sets",
    "PADERBORN": r"F:\Umar-Wisal-Work\Datasets\Paderborn",
}

OUTPUT_DIR = r"F:\Umar-Wisal-Work\UnifiedBenchmark"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================

WINDOW_SIZE = 4096
STEP_SIZE = 2048

# ============================================================
# HELPERS
# ============================================================

def sliding_windows(signal, win, step):
    windows = []

    for i in range(0, len(signal) - win, step):
        windows.append(signal[i:i+win])

    return windows


def normalize(x):
    x = np.asarray(x).astype(np.float32)
    return (x - np.mean(x)) / (np.std(x) + 1e-8)


# ============================================================
# CWRU LOADER
# ============================================================

def load_cwru():

    dataset_rows = []

    root = Path(ROOTS["CWRU"])

    mapping = {
        "healthy.mat": "normal",
        "inner.mat": "inner",
        "outer.mat": "outer",
        "ball.mat": "ball",
    }

    for file in root.glob("*.mat"):

        label = mapping[file.name]

        mat = sio.loadmat(file, squeeze_me=True)

        key = [k for k in mat.keys() if not k.startswith("__")][0]

        obj = mat[key]

        # Drive End only
        drive_end = obj[0, 0]

        for severity_idx, arr in enumerate(drive_end):

            arr = np.asarray(arr)

            if arr.ndim != 2:
                continue

            for sample_idx in range(arr.shape[0]):

                signal = arr[sample_idx]

                signal = normalize(signal)

                windows = sliding_windows(
                    signal,
                    WINDOW_SIZE,
                    STEP_SIZE
                )

                for win_idx, win in enumerate(windows):

                    save_name = (
                        f"CWRU_{label}_"
                        f"{severity_idx}_{sample_idx}_{win_idx}.npy"
                    )

                    save_path = os.path.join(
                        OUTPUT_DIR,
                        save_name
                    )

                    np.save(save_path, win)

                    dataset_rows.append({
                        "dataset": "CWRU",
                        "label": label,
                        "file": file.name,
                        "severity": severity_idx,
                        "sample_idx": sample_idx,
                        "window_idx": win_idx,
                        "rpm": None,
                        "fs": 12000,
                        "domain": "laboratory",
                        "fault_type": label,
                        "path": save_path
                    })

    return dataset_rows


# ============================================================
# HUST LOADER
# ============================================================

def infer_hust_label(name):

    stem = Path(name).stem.upper()

    if stem.startswith("N"):
        return "normal"

    if stem.startswith("I") and "O" not in stem and "B" not in stem:
        return "inner"

    if stem.startswith("O") and "B" not in stem:
        return "outer"

    if stem.startswith("B"):
        return "ball"

    if "IB" in stem:
        return "inner_ball"

    if "IO" in stem:
        return "inner_outer"

    if "OB" in stem:
        return "outer_ball"

    return "unknown"


def load_hust():

    dataset_rows = []

    root = Path(ROOTS["HUST"])

    for file in tqdm(list(root.glob("*.mat"))):

        label = infer_hust_label(file.name)

        mat = sio.loadmat(file, squeeze_me=True)

        signal = mat["data"].flatten()

        signal = normalize(signal)

        fs = float(mat["fs"])

        rpm = None

        if "rpm" in mat:
            rpm_arr = mat["rpm"].flatten()
            rpm = float(np.mean(rpm_arr))

        windows = sliding_windows(
            signal,
            WINDOW_SIZE,
            STEP_SIZE
        )

        for win_idx, win in enumerate(windows):

            save_name = (
                f"HUST_{label}_{file.stem}_{win_idx}.npy"
            )

            save_path = os.path.join(
                OUTPUT_DIR,
                save_name
            )

            np.save(save_path, win)

            dataset_rows.append({
                "dataset": "HUST",
                "label": label,
                "file": file.name,
                "severity": None,
                "sample_idx": None,
                "window_idx": win_idx,
                "rpm": rpm,
                "fs": fs,
                "domain": "laboratory",
                "fault_type": label,
                "path": save_path
            })

    return dataset_rows


# ============================================================
# MFPT LOADER
# ============================================================

def infer_mfpt_label(path):

    p = str(path).lower()

    if "baseline" in p:
        return "normal"

    if "outer" in p:
        return "outer"

    if "inner" in p:
        return "inner"

    return "unknown"


def load_mfpt():

    dataset_rows = []

    root = Path(ROOTS["MFPT"])

    files = list(root.rglob("*.mat"))

    for file in tqdm(files):

        label = infer_mfpt_label(file)

        mat = sio.loadmat(
            file,
            squeeze_me=True,
            struct_as_record=False
        )

        if "bearing" not in mat:
            continue

        bearing = mat["bearing"]

        signal = np.asarray(bearing.gs).flatten()

        signal = normalize(signal)

        fs = float(bearing.sr)

        windows = sliding_windows(
            signal,
            WINDOW_SIZE,
            STEP_SIZE
        )

        for win_idx, win in enumerate(windows):

            save_name = (
                f"MFPT_{label}_{file.stem}_{win_idx}.npy"
            )

            save_path = os.path.join(
                OUTPUT_DIR,
                save_name
            )

            np.save(save_path, win)

            dataset_rows.append({
                "dataset": "MFPT",
                "label": label,
                "file": file.name,
                "severity": None,
                "sample_idx": None,
                "window_idx": win_idx,
                "rpm": None,
                "fs": fs,
                "domain": "industrial",
                "fault_type": label,
                "path": save_path
            })

    return dataset_rows


# ============================================================
# PADERBORN LOADER
# ============================================================

def infer_paderborn_label(folder):

    folder = folder.upper()

    if folder.startswith("K0"):
        return "normal"

    if folder.startswith("KA"):
        return "outer"

    if folder.startswith("KI"):
        return "inner"

    if folder.startswith("KB"):
        return "combined"

    return "unknown"


def load_paderborn():

    dataset_rows = []

    root = Path(ROOTS["PADERBORN"])

    files = list(root.rglob("*.mat"))

    for file in tqdm(files):

        folder = file.parent.name

        label = infer_paderborn_label(folder)

        try:

            mat = sio.loadmat(
                file,
                squeeze_me=True,
                struct_as_record=False
            )

            key = [k for k in mat.keys() if not k.startswith("__")][0]

            obj = mat[key]

            Y = obj.Y

            vibration = None
            rpm = None

            for y in Y:

                name = str(y.Name).lower()

                if "vibration" in name:
                    vibration = np.asarray(y.Data).flatten()

                if "speed" in name:
                    rpm = np.mean(np.asarray(y.Data).flatten())

            # fallback
            if vibration is None:
                vibration = np.asarray(Y[-1].Data).flatten()

            vibration = normalize(vibration)

            windows = sliding_windows(
                vibration,
                WINDOW_SIZE,
                STEP_SIZE
            )

            for win_idx, win in enumerate(windows):

                save_name = (
                    f"PADERBORN_{label}_{folder}_"
                    f"{file.stem}_{win_idx}.npy"
                )

                save_path = os.path.join(
                    OUTPUT_DIR,
                    save_name
                )

                np.save(save_path, win)

                dataset_rows.append({
                    "dataset": "PADERBORN",
                    "label": label,
                    "file": file.name,
                    "bearing_id": folder,
                    "window_idx": win_idx,
                    "rpm": rpm,
                    "fs": 64000,
                    "domain": (
                        "real"
                        if folder in ["KA04", "KA15", "KI14"]
                        else "artificial"
                    ),
                    "fault_type": label,
                    "path": save_path
                })

        except Exception as e:
            print("ERROR:", file, e)

    return dataset_rows


# ============================================================
# MAIN
# ============================================================

all_rows = []

print("\nLoading CWRU...")
all_rows.extend(load_cwru())

print("\nLoading HUST...")
all_rows.extend(load_hust())

print("\nLoading MFPT...")
all_rows.extend(load_mfpt())

print("\nLoading Paderborn...")
all_rows.extend(load_paderborn())

df = pd.DataFrame(all_rows)

csv_path = os.path.join(
    OUTPUT_DIR,
    "unified_benchmark_metadata.csv"
)

df.to_csv(csv_path, index=False)

print("\nDONE")
print(df.head())

print("\nSaved metadata:")
print(csv_path)

print("\nTotal windows:", len(df))


Loading CWRU...

Loading HUST...


100%|██████████| 99/99 [12:18<00:00,  7.46s/it]



Loading MFPT...


100%|██████████| 24/24 [01:35<00:00,  3.96s/it]



Loading Paderborn...


 39%|███▉      | 992/2560 [1:02:35<2:38:29,  6.06s/it]

ERROR: F:\Umar-Wisal-Work\Datasets\Paderborn\KA08\N15_M01_F10_KA08_2.mat Expecting matrix here


100%|██████████| 2560/2560 [2:41:26<00:00,  3.78s/it]  



DONE
  dataset label      file  severity  sample_idx  window_idx  rpm       fs  \
0    CWRU  ball  ball.mat       0.0         0.0           0  NaN  12000.0   
1    CWRU  ball  ball.mat       0.0         0.0           1  NaN  12000.0   
2    CWRU  ball  ball.mat       0.0         0.0           2  NaN  12000.0   
3    CWRU  ball  ball.mat       0.0         0.0           3  NaN  12000.0   
4    CWRU  ball  ball.mat       0.0         1.0           0  NaN  12000.0   

       domain fault_type                                               path  \
0  laboratory       ball  F:\Umar-Wisal-Work\UnifiedBenchmark\CWRU_ball_...   
1  laboratory       ball  F:\Umar-Wisal-Work\UnifiedBenchmark\CWRU_ball_...   
2  laboratory       ball  F:\Umar-Wisal-Work\UnifiedBenchmark\CWRU_ball_...   
3  laboratory       ball  F:\Umar-Wisal-Work\UnifiedBenchmark\CWRU_ball_...   
4  laboratory       ball  F:\Umar-Wisal-Work\UnifiedBenchmark\CWRU_ball_...   

  bearing_id  
0        NaN  
1        NaN  
2        Na

In [4]:
# create_leakage_free_splits.py

import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

META = r"F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata.csv"
OUT = r"F:\Umar-Wisal-Work\UnifiedBenchmark\splits"
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(META)

# Clean group identifiers
df["bearing_id"] = df["bearing_id"].fillna("none")
df["file"] = df["file"].fillna("unknown")
df["group_id"] = (
    df["dataset"].astype(str) + "__" +
    df["bearing_id"].astype(str) + "__" +
    df["file"].astype(str)
)

# ------------------------------------------------------------
# Split 1: standard leakage-free file-level split
# ------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df["label"], groups=df["group_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=43)
tr_idx, val_idx = next(
    gss2.split(train_df, train_df["label"], groups=train_df["group_id"])
)

final_train = train_df.iloc[tr_idx]
final_val = train_df.iloc[val_idx]

final_train.to_csv(os.path.join(OUT, "file_level_train.csv"), index=False)
final_val.to_csv(os.path.join(OUT, "file_level_val.csv"), index=False)
test_df.to_csv(os.path.join(OUT, "file_level_test.csv"), index=False)

# ------------------------------------------------------------
# Split 2: leave-one-dataset-out
# ------------------------------------------------------------
for target in sorted(df["dataset"].unique()):
    train = df[df["dataset"] != target]
    test = df[df["dataset"] == target]

    train.to_csv(os.path.join(OUT, f"LODO_train_except_{target}.csv"), index=False)
    test.to_csv(os.path.join(OUT, f"LODO_test_{target}.csv"), index=False)

# ------------------------------------------------------------
# Split 3: Paderborn artificial-to-real
# ------------------------------------------------------------
pb = df[df["dataset"] == "PADERBORN"].copy()

pb_art = pb[pb["domain"] == "artificial"]
pb_real = pb[pb["domain"] == "real"]

pb_art.to_csv(os.path.join(OUT, "paderborn_artificial_train.csv"), index=False)
pb_real.to_csv(os.path.join(OUT, "paderborn_real_test.csv"), index=False)

# ------------------------------------------------------------
# Split 4: cross-dataset harsh test
# Train public lab datasets, test Paderborn real
# ------------------------------------------------------------
train = df[
    (df["dataset"].isin(["CWRU", "HUST", "MFPT"])) |
    ((df["dataset"] == "PADERBORN") & (df["domain"] == "artificial"))
]

test = df[
    (df["dataset"] == "PADERBORN") &
    (df["domain"] == "real")
]

train.to_csv(os.path.join(OUT, "harsh_train_lab_plus_artificial.csv"), index=False)
test.to_csv(os.path.join(OUT, "harsh_test_paderborn_real.csv"), index=False)

print("DONE")
print("Total:", len(df))
print("Train:", len(final_train))
print("Val:", len(final_val))
print("Test:", len(test_df))
print("Splits saved to:", OUT)

DONE
Total: 347707
Train: 237802
Val: 41298
Test: 68607
Splits saved to: F:\Umar-Wisal-Work\UnifiedBenchmark\splits


In [5]:
# pack_windows_to_hdf5.py

import os
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

META = r"F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata.csv"
OUT_H5 = r"F:\Umar-Wisal-Work\UnifiedBenchmark\bearing_windows_4096.h5"
OUT_META = r"F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata_h5.csv"

df = pd.read_csv(META)

label_encoder = LabelEncoder()
domain_encoder = LabelEncoder()
dataset_encoder = LabelEncoder()

df["y"] = label_encoder.fit_transform(df["label"].astype(str))
df["domain_id"] = domain_encoder.fit_transform(df["dataset"].astype(str) + "_" + df["domain"].astype(str))
df["dataset_id"] = dataset_encoder.fit_transform(df["dataset"].astype(str))

print("Labels:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print("Domains:", dict(zip(domain_encoder.classes_, domain_encoder.transform(domain_encoder.classes_))))

n = len(df)
win = 4096

with h5py.File(OUT_H5, "w") as h5:
    X = h5.create_dataset(
        "X",
        shape=(n, win),
        dtype="float32",
        chunks=(512, win),
        compression="gzip",
        compression_opts=4,
    )

    y = h5.create_dataset("y", data=df["y"].values.astype("int64"))
    d = h5.create_dataset("domain_id", data=df["domain_id"].values.astype("int64"))
    ds = h5.create_dataset("dataset_id", data=df["dataset_id"].values.astype("int64"))

    for i, p in enumerate(tqdm(df["path"].values)):
        X[i] = np.load(p).astype("float32")

df["h5_index"] = np.arange(n)
df.to_csv(OUT_META, index=False)

print("Saved:", OUT_H5)
print("Saved:", OUT_META)

Labels: {'ball': np.int64(0), 'combined': np.int64(1), 'inner': np.int64(2), 'inner_ball': np.int64(3), 'inner_outer': np.int64(4), 'normal': np.int64(5), 'outer': np.int64(6), 'outer_ball': np.int64(7), 'unknown': np.int64(8)}
Domains: {'CWRU_laboratory': np.int64(0), 'HUST_laboratory': np.int64(1), 'MFPT_industrial': np.int64(2), 'PADERBORN_artificial': np.int64(3), 'PADERBORN_real': np.int64(4)}


100%|██████████| 347707/347707 [8:18:39<00:00, 11.62it/s]  


Saved: F:\Umar-Wisal-Work\UnifiedBenchmark\bearing_windows_4096.h5
Saved: F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata_h5.csv


In [6]:
# clean_h5_metadata.py

import pandas as pd
from sklearn.preprocessing import LabelEncoder

META = r"F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata_h5.csv"
OUT = r"F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata_h5_clean.csv"

df = pd.read_csv(META)

df = df[df["label"] != "unknown"].copy()
df = df[df["label"].isin([
    "normal", "inner", "outer", "ball",
    "combined", "inner_ball", "inner_outer", "outer_ball"
])].copy()

le = LabelEncoder()
df["y_clean"] = le.fit_transform(df["label"].astype(str))

print("Clean labels:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Remaining samples:", len(df))

df.to_csv(OUT, index=False)
print("Saved:", OUT)

Clean labels: {'ball': np.int64(0), 'combined': np.int64(1), 'inner': np.int64(2), 'inner_ball': np.int64(3), 'inner_outer': np.int64(4), 'normal': np.int64(5), 'outer': np.int64(6), 'outer_ball': np.int64(7)}
Remaining samples: 347115
Saved: F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata_h5_clean.csv


In [7]:
# train_masked_spectrum_pretraining.py

import os
import math
import h5py
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# ============================================================
# PATHS
# ============================================================

H5_PATH = r"F:\Umar-Wisal-Work\UnifiedBenchmark\bearing_windows_4096.h5"
META_PATH = r"F:\Umar-Wisal-Work\UnifiedBenchmark\unified_benchmark_metadata_h5_clean.csv"
OUT_DIR = r"F:\Umar-Wisal-Work\UnifiedBenchmark\pretraining_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# CONFIG
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-4

N_FFT = 4096
SPEC_LEN = N_FFT // 2 + 1
PATCH_SIZE = 16
N_PATCHES = math.ceil(SPEC_LEN / PATCH_SIZE)

D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1
MASK_RATIO = 0.35

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Device:", DEVICE)
print("Spectrum length:", SPEC_LEN)
print("Patches:", N_PATCHES)

# ============================================================
# DATASET
# ============================================================

class MaskedSpectrumDataset(Dataset):
    def __init__(self, meta_path, h5_path):
        self.df = pd.read_csv(meta_path).reset_index(drop=True)
        self.h5_path = h5_path
        self.h5 = None

    def __len__(self):
        return len(self.df)

    def _open_h5(self):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, "r")

    def __getitem__(self, idx):
        self._open_h5()

        h5_idx = int(self.df.iloc[idx]["h5_index"])
        x = self.h5["X"][h5_idx].astype(np.float32)

        # Time-domain normalization
        x = (x - x.mean()) / (x.std() + 1e-8)

        # FFT magnitude spectrum
        spec = np.abs(np.fft.rfft(x, n=N_FFT)).astype(np.float32)

        # log compression
        spec = np.log1p(spec)

        # spectrum normalization
        spec = (spec - spec.mean()) / (spec.std() + 1e-8)

        # pad to patch length
        pad_len = N_PATCHES * PATCH_SIZE - len(spec)
        if pad_len > 0:
            spec = np.pad(spec, (0, pad_len))

        spec = spec.reshape(N_PATCHES, PATCH_SIZE)

        return torch.tensor(spec, dtype=torch.float32)

# ============================================================
# MODEL
# ============================================================

class MaskedSpectrumTransformer(nn.Module):
    def __init__(
        self,
        patch_size=PATCH_SIZE,
        n_patches=N_PATCHES,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.patch_embed = nn.Linear(patch_size, d_model)

        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.pos_embed = nn.Parameter(
            torch.randn(1, n_patches, d_model) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.decoder = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, patch_size)
        )

    def forward(self, x, mask):
        """
        x:    [B, P, patch_size]
        mask: [B, P], True means masked
        """

        z = self.patch_embed(x)
        z = z + self.pos_embed[:, :z.size(1), :]

        mask_token = self.mask_token.expand(z.size(0), z.size(1), -1)
        z = torch.where(mask.unsqueeze(-1), mask_token, z)

        h = self.encoder(z)
        recon = self.decoder(h)

        return recon, h

# ============================================================
# MASKING
# ============================================================

def make_random_mask(batch_size, n_patches, mask_ratio):
    mask = torch.zeros(batch_size, n_patches, dtype=torch.bool)

    n_mask = int(n_patches * mask_ratio)

    for i in range(batch_size):
        ids = torch.randperm(n_patches)[:n_mask]
        mask[i, ids] = True

    return mask

# ============================================================
# TRAIN
# ============================================================

dataset = MaskedSpectrumDataset(META_PATH, H5_PATH)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

model = MaskedSpectrumTransformer().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.MSELoss()

best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    model.train()
    losses = []

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for x in pbar:
        x = x.to(DEVICE)

        mask = make_random_mask(
            x.size(0),
            x.size(1),
            MASK_RATIO
        ).to(DEVICE)

        recon, _ = model(x, mask)

        loss = criterion(recon[mask], x[mask])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=np.mean(losses))

    epoch_loss = float(np.mean(losses))

    ckpt = {
        "model": model.state_dict(),
        "epoch": epoch,
        "loss": epoch_loss,
        "config": {
            "patch_size": PATCH_SIZE,
            "n_patches": N_PATCHES,
            "d_model": D_MODEL,
            "n_heads": N_HEADS,
            "n_layers": N_LAYERS,
            "mask_ratio": MASK_RATIO,
            "n_fft": N_FFT,
        }
    }

    torch.save(
        ckpt,
        os.path.join(OUT_DIR, "last_masked_spectrum_transformer.pt")
    )

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(
            ckpt,
            os.path.join(OUT_DIR, "best_masked_spectrum_transformer.pt")
        )

    print(f"Epoch {epoch}: loss={epoch_loss:.6f}, best={best_loss:.6f}")

print("Pretraining complete.")
print("Saved to:", OUT_DIR)

Device: cuda
Spectrum length: 2049
Patches: 129


C:\Users\Muhammad Umar\AppData\Roaming\Python\Python313\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Epoch 1/50: 100%|██████████| 2711/2711 [1:52:43<00:00,  2.49s/it, loss=0.989]


Epoch 1: loss=0.988784, best=0.988784


Epoch 2/50: 100%|██████████| 2711/2711 [1:50:52<00:00,  2.45s/it, loss=0.987]


Epoch 2: loss=0.986761, best=0.986761


...

Epoch 48: loss=0.986192, best=0.985853


Epoch 49/50: 100%|██████████| 2711/2711 [1:50:51<00:00,  2.45s/it, loss=0.986]  


Epoch 49: loss=0.986248, best=0.985853


Epoch 50/50: 100%|██████████| 2711/2711 [1:51:16<00:00,  2.46s/it, loss=0.986]

Epoch 50: loss=0.985991, best=0.985853
Pretraining complete.
Saved to: F:\Umar-Wisal-Work\UnifiedBenchmark\pretraining_outputs
